# SETUP AND INSTALLATION

In [ ]:
!apt-get install -y zstd
!pip -q install langchain langchain-community langchain-core ollama requests pillow langchain-ollama
!pip -q install langchain langchain-core langchain-ollama
!pip -q install ddgs wikipedia
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 52 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (834 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 60.8 MB/s eta

In [ ]:
# Start Ollama server (for Colab)
import subprocess
import threading
import time
import os

def start_ollama_server():
    """Start Ollama server in background"""
    process = subprocess.Popen(['ollama', 'serve'],
                             stdout=subprocess.DEVNULL,
                             stderr=subprocess.STDOUT)
    return process

# Start Ollama in background
print("🔄 Starting Ollama server...")
ollama_process = start_ollama_server()
time.sleep(5)  # Give it time to start

print("✅ Setup completed successfully!")

🔄 Starting Ollama server...
✅ Setup completed successfully!


In [ ]:
!ollama pull llama3.2:latest

# LESSON 1: FOUNDATIONAL MODELS

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

response = llm.invoke("What is the capital of France?")
print(f"Question: What is the capital of France?")
print(f"Answer: {response}")

# Create a simple chain for basic chat
prompt = ChatPromptTemplate.from_template("Answer the following question: {question}")
chain = prompt | llm | StrOutputParser()

# Test the chain
question = "Explain what AI agents are in simple terms."
answer = chain.invoke({"question": question})
print(f"\nQuestion: {question}")
print(f"Answer: {answer}")

ConnectError: [Errno 111] Connection refused

# Tools

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import WikipediaQueryRun
from langchain_core.tools import tool
from datetime import datetime

# 1. Search Tool
search_tool = DuckDuckGoSearchRun()

# 2. Wikipedia Tool
wikipedia = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper()
)

# 3. Custom Recipe Tool
def recipe_search(ingredients: str) -> str:
    """Search for recipes based on ingredients."""

    recipes_db = {
        "chicken, rice": [
            "Chicken and Rice Pilaf",
            "One-Pot Chicken and Rice",
            "Chicken Fried Rice"
          ],
        "pasta, tomato": [
            "Classic Tomato Pasta",
            "Pasta with Tomato Sauce",
            "Spaghetti with Marinara"
        ],
        "beef, vegetables": [
            "Beef and Vegetable Stew",
            "Stir-Fried Beef with Vegetables",
            "Beef Vegetable Soup"
        ],
        "fish, lemon": [
            "Lemon Baked Fish",
            "Fish with Lemon Sauce",
            "Grilled Fish with Lemon"
        ]
    }

    for key, recipes in recipes_db.items():
        key_ingredients = set(key.split(", "))
        user_ingredients = set(
            item.strip().lower()
            for item in ingredients.split(",")
        )

        if user_ingredients.issubset(key_ingredients):
            return f"Found recipes for {ingredients}: {', '.join(recipes)}"

    return f"No specific recipes found for {ingredients}. Try different ingredients."


@tool
def recipe_finder(ingredients: str) -> str:
    """Search for recipes based on available ingredients.
    Input should be a comma-separated list of ingredients."""
    return recipe_search(ingredients)


@tool
def current_time() -> str:
    """Get the current date and time."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


# List all tools
tools = [
    search_tool,
    wikipedia,
    recipe_finder,
    current_time
]

tool_names = [t.name for t in tools]

print(f"📋 Available tools: {tool_names}")
print("✅ Tools initialized successfully!")


# Test recipe tool
print("\n1. Recipe Finder Test:")
result = recipe_finder.invoke("chicken, rice")
print(result)


# Test time tool
print("\n2. Current Time Test:")
result = current_time.invoke({})
print(result)

📋 Available tools: ['duckduckgo_search', 'wikipedia', 'recipe_finder', 'current_time']
✅ Tools initialized successfully!

1. Recipe Finder Test:
Found recipes for chicken, rice: Chicken and Rice Pilaf, One-Pot Chicken and Rice, Chicken Fried Rice

2. Current Time Test:
2026-09-21 08:47:37


#  LESSON 3: SHORT-TERM MEMORY

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

# Store conversation history
chat_history = []

print("🔧 Initializing conversation memory...")
print("✅ Memory-based conversation ready!")


def chat_with_memory(user_input):
    global chat_history

    # Add user's message
    chat_history.append(HumanMessage(content=user_input))

    # Send entire conversation to the model
    response = llm.invoke(chat_history)

    # Store AI response
    chat_history.append(response) # Corrected line: directly append the AIMessage object

    return response


# Test multi-turn conversation
print("\n🧪 Testing multi-turn conversation with memory:")

# First turn
response1 = chat_with_memory(
    "Hello! My name is John and I like Italian food."
)
print(f"AI: {response1}")

# Second turn
response2 = chat_with_memory(
    "What did I say my name was?"
)
print(f"AI: {response2}")

# Third turn
response3 = chat_with_memory(
    "What kind of food did I mention I like?"
)
print(f"AI: {response3}")

🔧 Initializing conversation memory...
✅ Memory-based conversation ready!

🧪 Testing multi-turn conversation with memory:
AI: content="Ciao John! Nice to meet you! Italian food is definitely a popular and delicious choice. From pasta dishes like spaghetti carbonara and fettuccine Alfredo, to pizzas, risottos, and gelatos, there's something for everyone. What's your favorite Italian dish?" additional_kwargs={} response_metadata={'model': 'llama3.2', 'created_at': '2026-09-21T08:47:50.462124802Z', 'done': True, 'done_reason': 'stop', 'total_duration': 827170928, 'load_duration': 2685896, 'prompt_eval_count': 37, 'prompt_eval_duration': 38727000, 'eval_count': 59, 'eval_duration': 761670000, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'} id='lc_run--01a0c326-6981-78a0-b8fd-2cc256f6a835-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 37, 'output_tokens': 59, 'total_tokens': 96}
AI: content='You said your name is John.' additional_kwargs={} res

# LESSON 4: MULTIMODAL MESSAGES

In [ ]:
from PIL import Image
import base64
from io import BytesIO
import requests
import tempfile

# Simulate image analysis tool (for demonstration)
@tool
def image_analyzer(image_url: str) -> str:
    """Analyze an image and describe what it contains.
    Input should be a URL to an image."""

    try:
        # Download image from URL
        response = requests.get(image_url, timeout=30)
        response.raise_for_status()

        # Save temporarily
        with tempfile.NamedTemporaryFile(
            suffix=".jpg",
            delete=False
        ) as temp_file:

            temp_file.write(response.content)
            image_path = temp_file.name

        # Send image to Ollama Vision model
        result = ollama.chat(
            model="qwen2.5vl:3b",
            messages=[
                {
                    "role": "user",
                    "content": """
                    Analyze this food image.

                    Identify:
                    Most likely dish

                    Do not invent ingredients that cannot reasonably
                    be identified from the image.
                    """,
                    "images": [image_path]
                }
            ]
        )

        # Delete temporary file
        os.remove(image_path)

        return result["message"]["content"]

    except Exception as e:
        return f"Image analysis failed: {str(e)}"

# Add multimodal tool to our tools list
tools.append(image_analyzer)
print("📋 Updated tools:", [tool.name for tool in tools])

📋 Updated tools: ['duckduckgo_search', 'wikipedia', 'recipe_finder', 'current_time', 'image_analyzer', 'image_analyzer', 'image_analyzer']


In [ ]:
!ollama pull qwen2.5vl:3b

In [ ]:
image_path = "/content/cake.jpeg"

In [ ]:
import ollama

response = ollama.chat(
    model="qwen2.5vl:3b",
    messages=[
        {
            "role": "user",
            "content": """
            Analyze this food image.

            Identify:
            - The most likely dish
            - Visible ingredients
            - How confident you are
            - Any alternative possibilities

            Do not invent ingredients that cannot reasonably be seen.
            """,
            "images": [image_path]
        }
    ]
)

print(response["message"]["content"])

### Analysis of the Food Image

#### Most Likely Dish:
- **Dish:** Vanilla Layer Cake with Sprinkles

#### Visible Ingredients:
- Vanilla cake layers
- Vanilla buttercream frosting
- Colorful sprinkles

#### Confidence:
- **Confidence:** High
  - The image clearly shows a layered cake with visible layers of vanilla cake and vanilla frosting. The presence of colorful sprinkles on the top and sides of the cake is a strong indicator of a vanilla cake with sprinkles.

#### Alternative Possibilities:
- **Vanilla Layer Cake with Rainbow Sprinkles:** This is a common variation where the sprinkles are in a rainbow pattern.
- **Vanilla Layer Cake with Confetti Sprinkles:** This variation uses a mix of different colors in a confetti-like pattern.
- **Vanilla Layer Cake with Edible Glitter:** This variation uses edible glitter instead of sprinkles, adding a sparkly effect.

### Summary
The image depicts a vanilla layer cake with colorful sprinkles. The cake is likely a vanilla layer cake with spr

# PROJECT: PERSONAL CHEF

In [ ]:
from langchain.agents import create_agent

# ============================================================
# 🤖 CREATE PERSONAL CHEF AGENT
# ============================================================

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""You are a helpful Personal Chef AI assistant.
                    You can:
                    - Find recipes based on ingredients.
                    - Search the web for cooking information.
                    - Search Wikipedia for general information.
                    - Answer general cooking questions.
                    - Remember relevant information from the current conversation.
                    - If an image url is provided use the image analyzer and give a recipe based on the image description.

                    Use the available tools when they are useful.

                    Give clear, practical answers.
                    Do not use tools when they are unnecessary.
                    """
                    )

print("✅ Personal Chef agent ready!")


# ============================================================
# 🧠 CONVERSATION MEMORY
# ============================================================

conversation_history = []


def chat_with_chef(user_input):
    """Send a message to the chef while maintaining conversation history."""

    # Add user message
    conversation_history.append({
        "role": "user",
        "content": user_input
    })

    # Send conversation to agent
    response = agent.invoke({
        "messages": conversation_history
    })

    # Replace history with the complete conversation returned
    conversation_history.clear()
    conversation_history.extend(response["messages"])

    # Return final AI message
    return response["messages"][-1].content


# ============================================================
# 🍳 INTERACTIVE PERSONAL CHEF
# ============================================================

def personal_chef_agent():
    """Interactive Personal Chef Agent"""

    print("\n" + "🍳" + "=" * 50 + "🍳")
    print("        WELCOME TO YOUR PERSONAL CHEF AGENT")
    print("🍳" + "=" * 50 + "🍳")

    print("\nI can help you with:")
    print("1. Finding recipes based on ingredients")
    print("2. Searching for cooking information")
    print("3. Answering general cooking questions")
    print("4. Remembering information from our conversation")

    print("\nType 'quit' to exit")
    print("-" * 60)

    while True:

        try:
            user_input = input("\n👨‍🍳 You: ").strip()

            if user_input.lower() in ["quit", "exit", "q"]:
                print("\n👋 Thank you for using Personal Chef Agent!")
                print("Goodbye!")
                break

            if not user_input:
                continue

            print("\n🔄 Thinking...")

            final_answer = chat_with_chef(user_input)

            print(f"\n🦾 Chef: {final_answer}")

        except KeyboardInterrupt:
            print("\n\n👋 Goodbye!")
            break

        except Exception as e:
            print(f"\n❌ Error: {str(e)}")
            print("Please try again.")


# ============================================================
# 🧪 COMPREHENSIVE TESTS
# ============================================================

def run_comprehensive_tests():

    print("\n" + "=" * 60)
    print("🧪 RUNNING COMPREHENSIVE TESTS")
    print("=" * 60)

    tests = [
        {
            "name": "Basic Model Test",
            "input": "What is 2+2?",
            "expected": "4"
        },
        {
            "name": "Recipe Tool Test",
            "input": "Find recipes with chicken and rice",
            "expected": "Chicken"
        },
        {
            "name": "Search Tool Test",
            "input": "Search for best pasta recipes",
            "expected": "pasta"
        },
        {
            "name": "Image Analysis Test",
            "input": "Give me the recipe for this image: https://sugarandsparrow.s3.us-west-2.amazonaws.com/flour/wp-content/uploads/2022/02/16215641/Best-Vanilla-Cake-Recipe-header.jpeg",
            "expected": "Vanilla cake with sprinkles"
        }
    ]

    print("🧪 Running automated tests...")

    for test in tests:

        try:
            print(f"\n📋 Test: {test['name']}")
            print(f"❓ Question: {test['input']}")

            response = agent.invoke({
                "messages": [
                    {
                        "role": "user",
                        "content": test["input"]
                    }
                ]
            })

            output = response["messages"][-1].content

            success = (
                test["expected"].lower()
                in output.lower()
            )

            status = "✅ PASS" if success else "❌ FAIL"

            print(f"📝 Response: {output[:200]}...")
            print(f"📊 Status: {status}")

        except Exception as e:
            print(f"❌ Test failed: {e}")


# ============================================================
# 📊 PERFORMANCE MONITORING
# ============================================================

def monitor_performance():

    print("\n" + "=" * 60)
    print("📊 PERFORMANCE MONITORING")
    print("=" * 60)

    import psutil
    import platform

    print(
        f"🖥️ System: "
        f"{platform.system()} {platform.release()}"
    )

    print(
        f"💾 CPU Usage: "
        f"{psutil.cpu_percent()}%"
    )

    print(
        f"🧠 Memory Usage: "
        f"{psutil.virtual_memory().percent}%"
    )

    # Ollama information
    if hasattr(llm, "model"):
        print(f"🦾 Model: {llm.model}")

    if hasattr(llm, "temperature"):
        print(f"🌡️ Temperature: {llm.temperature}")

    print(f"🔧 Tools Available: {len(tools)}")

    print(
        f"💬 Conversation Messages: "
        f"{len(conversation_history)}"
    )


# ============================================================
# 🎯 MAIN EXECUTION
# ============================================================

if __name__ == "__main__":

    # Run tests
    run_comprehensive_tests()

    # Show performance
    monitor_performance()

    # Start Personal Chef
    print("\n" + "🎉" + "=" * 50 + "🎉")
    print("        ALL SYSTEMS READY!")
    print("🎉" + "=" * 50 + "🎉")

    personal_chef_agent()

✅ Personal Chef agent ready!

🧪 RUNNING COMPREHENSIVE TESTS
🧪 Running automated tests...

📋 Test: Basic Model Test
❓ Question: What is 2+2?


/usr/local/lib/python3.13/dist-packages/wikipedia/wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file /usr/local/lib/python3.13/dist-packages/wikipedia/wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')


📝 Response: The answer to 2+2 is 4, not 5. This is a basic arithmetic fact that is widely accepted and used in mathematics. The phrase "2 + 2 = 5" is often used as an example of a logical error or a mathematical ...
📊 Status: ✅ PASS

📋 Test: Recipe Tool Test
❓ Question: Find recipes with chicken and rice
📝 Response: Here are three simple recipes you can try:

**Recipe 1: Chicken and Rice Pilaf**

Ingredients:

* 1 lb boneless, skinless chicken breast or thighs
* 2 cups uncooked white rice
* 4 cups chicken broth
*...
📊 Status: ✅ PASS

📋 Test: Search Tool Test
❓ Question: Search for best pasta recipes
📝 Response: Here are three delicious pasta recipes for you to try:

**Recipe 1: Classic Tomato Pasta**

Ingredients:

* 12 oz pasta of your choice
* 2 cups cherry tomatoes, halved
* 1 cup fresh basil leaves, chop...
📊 Status: ✅ PASS

📋 Test: Image Analysis Test
❓ Question: Give me the recipe for this image: https://sugarandsparrow.s3.us-west-2.amazonaws.com/flour/wp-content/uploads/2022/02/